<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TOPO_GPT_CASE6_corrected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CASE6 (corrected)

Same pipeline as before: frozen `openai/gpt-oss-20b`, three task heads trained A → B → C with `TopoAdamW` inside a Hugging Face `Trainer`, then publish and reload. Changes:

1. **Heads are float32** with a LayerNorm in front, zero-initialised, learning rate 1e-3 (was bfloat16 at 1e-5).
2. **Evaluation on the test split after every task**, giving an accuracy table and a measured forgetting score.
3. **One task/label table** (`TASKS`) used by training, the checkpoint and the inference cell.
4. **Digest recomputed from the loaded weights** in the final gate and in the reload cell.
5. **Last-token pooling works for left or right padding**; `Trainer` seed set to 123; `bf16` autocast off.
6. **Model card built from the measured numbers** in the checkpoint.

Run the cells in order.

In [ ]:
!nvidia-smi

Fri Oct  9 11:44:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/topogpt/gpt_oss_20b_hf_topo_run

In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/gpt_oss_20b_hf_topo_run

total 46K
-rw------- 1 root root  38K Oct  9 10:50 certified_topological_best.pt
drwx------ 2 root root 4.0K Oct  9 10:47 .
drwx------ 3 root root 4.0K Oct  9 10:40 ..


### 1. Train A → B → C, evaluate after each task, run the integrity gate, save the heads

In [ ]:
import hashlib
import math
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    logging as hf_logging,
)

hf_logging.set_verbosity_error()

# =====================================================================
# 1. GLOBAL CONFIGURATION & REPRODUCIBILITY SEED
# =====================================================================
SEED = 123
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

MODEL_ID = 'openai/gpt-oss-20b'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
SAFETY_CONSTANT = 0.9785142874

RUN_DIR = "/content/drive/MyDrive/topogpt/gpt_oss_20b_hf_topo_run"
CHECKPOINT_PATH = os.path.join(RUN_DIR, "certified_topological_best.pt")

MAX_LENGTH = 64
NUM_EPOCHS = 5
HEAD_LR = 1e-3          # FIX: was 1e-5, too small for a freshly initialised head
TRAIN_SPLIT = "train[:3000]"
TEST_SPLIT = "test[:2000]"

# FIX: one definition of the tasks, used for training, evaluation, the
# checkpoint and the inference cell. AG News: 0 World, 1 Sports, 2 Business,
# 3 Sci/Tech. 'labels' are the AG News ids mapped to head classes 0 and 1.
TASK_ORDER = ['A', 'B', 'C']
TASKS = {
    'A': {'labels': [0, 1], 'names': ['World', 'Sports']},
    'B': {'labels': [2, 3], 'names': ['Business', 'Sci/Tech']},
    'C': {'labels': [0, 3], 'names': ['World', 'Sci/Tech']},
}


def anchor_digest(embedding_layer, anchors):
    """SHA-256 over the anchor rows of the embedding matrix, read from the live weights."""
    w = embedding_layer.weight.data
    rows = [w[i] for i in anchors if i < w.shape[0]]
    blob = torch.cat([r.flatten().float() for r in rows]).detach().cpu().numpy().tobytes()
    return hashlib.sha256(blob).hexdigest()


# =====================================================================
# 2. TOPO-CBP CUSTOM OPTIMIZER CLASS (unchanged from the original)
# =====================================================================
class TopoAdamW(Optimizer):
    def __init__(self, params, lr=1e-5, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2, amsgrad=False, embedding_layer=None,
                 prime_anchors=None, lambda_euler=SAFETY_CONSTANT):
        if embedding_layer is None:
            raise ValueError("Topological Governor requires 'embedding_layer' to be explicitly passed.")

        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, amsgrad=amsgrad)
        super(TopoAdamW, self).__init__(params, defaults)

        self.embedding_layer = embedding_layer
        self.prime_anchors = prime_anchors or PRIME_ANCHORS
        self.lambda_euler = lambda_euler

        weight_data = self.embedding_layer.weight.data
        vocab_size = weight_data.shape[0]
        self.valid_anchors = [idx for idx in self.prime_anchors if idx < vocab_size]

        self.baseline_invariant_tensor_map = {
            idx: weight_data[idx].detach().clone() for idx in self.valid_anchors
        }
        self.certified_sha256_digest = anchor_digest(self.embedding_layer, self.valid_anchors)
        print(f"[TopoAdamW] Baseline Bedrock Locked. Certified Digest: {self.certified_sha256_digest[:16]}")

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        # GOVERNOR PHASE I: pre-step gradient clamping on the anchor rows
        if self.embedding_layer.weight.grad is not None:
            grad = self.embedding_layer.weight.grad
            grad.mul_(self.lambda_euler)
            for idx in self.valid_anchors:
                grad[idx].zero_()

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue

                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    if group['amsgrad']:
                        state['max_exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                if group['amsgrad']:
                    max_exp_avg_sq = state['max_exp_avg_sq']
                beta1, beta2 = group['betas']

                state['step'] += 1

                p.data.mul_(1.0 - group['lr'] * group['weight_decay'])
                exp_avg.mul_(beta1).add_(p.grad, alpha=1.0 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(p.grad, p.grad, value=1.0 - beta2)

                if group['amsgrad']:
                    torch.max(max_exp_avg_sq, exp_avg_sq, out=max_exp_avg_sq)
                    denom = (max_exp_avg_sq.sqrt() / math.sqrt(1.0 - beta2 ** state['step'])).add_(group['eps'])
                else:
                    denom = (exp_avg_sq.sqrt() / math.sqrt(1.0 - beta2 ** state['step'])).add_(group['eps'])

                step_size = group['lr'] / (1.0 - beta1 ** state['step'])
                p.addcdiv_(exp_avg, denom, value=-step_size)

        # GOVERNOR PHASE II: post-step restore of the anchor rows
        weight_data = self.embedding_layer.weight.data
        for idx, original_tensor in self.baseline_invariant_tensor_map.items():
            weight_data[idx].copy_(original_tensor)

        return loss

    def audit_telemetry(self):
        weight_data = self.embedding_layer.weight.data
        max_drift = 0.0
        for idx, original_tensor in self.baseline_invariant_tensor_map.items():
            drift = torch.max(torch.abs(weight_data[idx] - original_tensor)).item()
            max_drift = max(max_drift, drift)
        current_hash = anchor_digest(self.embedding_layer, self.valid_anchors)
        return max_drift, current_hash, (current_hash == self.certified_sha256_digest)


# =====================================================================
# 3. TASK-AWARE MODEL WRAPPER
# =====================================================================
def make_head(hidden_size):
    """FIX: float32 head. The original head was bfloat16, which has about 8 bits
    of precision, so Adam updates of 1e-5 were rounded away on most weights.
    LayerNorm brings the backbone's large hidden-state norms to unit scale, and
    the zero-initialised Linear makes the starting loss exactly ln(2) = 0.693."""
    head = nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, 2))
    nn.init.zeros_(head[1].weight)
    nn.init.zeros_(head[1].bias)
    return head.float()


def pool_last_token(hidden_states, attention_mask):
    """FIX: hidden state of the last real token, correct for left or right padding."""
    if attention_mask is None:
        return hidden_states[:, -1, :]
    positions = torch.arange(attention_mask.shape[1], device=attention_mask.device)
    last_idx = (attention_mask.long() * positions).argmax(dim=1)
    batch_idx = torch.arange(hidden_states.shape[0], device=hidden_states.device)
    return hidden_states[batch_idx, last_idx.to(hidden_states.device), :]


class GPTOSS_TaskAwareModel(nn.Module):
    def __init__(self, base_model, hidden_size, device=DEVICE):
        super().__init__()
        self.base_model = base_model
        self.hidden_size = hidden_size
        for task in TASK_ORDER:
            setattr(self, f'classifier_{task}', make_head(hidden_size).to(device))
        self.current_task = TASK_ORDER[0]

    def switch_task(self, task):
        assert task in TASK_ORDER, f"Unknown task: {task}"
        self.current_task = task

    def forward(self, input_ids, attention_mask=None, labels=None):
        with torch.no_grad():  # the backbone is frozen
            outputs = self.base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                return_dict=True,
            )
            pooled = pool_last_token(outputs.hidden_states[-1], attention_mask)

        head = getattr(self, f'classifier_{self.current_task}')
        head_device = head[1].weight.device
        # FIX: run the head in float32 with autocast off, whatever the backbone dtype
        with torch.autocast(device_type=head_device.type, enabled=False):
            logits = head(pooled.to(device=head_device, dtype=torch.float32))

        out = {"logits": logits}
        if labels is not None:
            out = {"loss": F.cross_entropy(logits, labels.to(head_device)), "logits": logits}
        return out


# =====================================================================
# 4. HUGGING FACE TRAINER WITH EMBEDDED TOPOADAMW
# =====================================================================
def unwrap_optimizer(optimizer):
    """Peel off Accelerate's AcceleratedOptimizer wrapper(s)."""
    while hasattr(optimizer, "optimizer"):
        optimizer = optimizer.optimizer
    return optimizer


class TopoTrainer(Trainer):
    def __init__(self, *args, embedding_layer=None, prime_anchors=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.embedding_layer = embedding_layer
        self.prime_anchors = prime_anchors or PRIME_ANCHORS

    def create_optimizer(self, model=None):
        if self.optimizer is None:
            opt_model = self.model if model is None else model
            decay_parameters = self.get_decay_parameter_names(opt_model)
            optimizer_grouped_parameters = [
                {
                    "params": [p for n, p in opt_model.named_parameters()
                               if (n in decay_parameters and p.requires_grad)],
                    "weight_decay": self.args.weight_decay,
                },
                {
                    "params": [p for n, p in opt_model.named_parameters()
                               if (n not in decay_parameters and p.requires_grad)],
                    "weight_decay": 0.0,
                },
            ]
            self.optimizer = TopoAdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(self.args.adam_beta1, self.args.adam_beta2),
                eps=self.args.adam_epsilon,
                weight_decay=self.args.weight_decay,
                embedding_layer=self.embedding_layer,
                prime_anchors=self.prime_anchors,
                lambda_euler=SAFETY_CONSTANT,
            )
        return self.optimizer


class TopoTelemetryCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        optimizer = kwargs.get("optimizer")
        if optimizer is None:
            return
        actual_opt = unwrap_optimizer(optimizer)
        if not hasattr(actual_opt, "audit_telemetry"):
            raise RuntimeError("Telemetry audit skipped: optimizer is not TopoAdamW.")
        max_drift, digest, hash_matched = actual_opt.audit_telemetry()
        status = "100% PASS" if hash_matched and max_drift == 0.0 else "VIOLATION"
        print(f"\n[TOPO Telemetry] Epoch {state.epoch:.0f} | Drift: {max_drift:.10f} | "
              f"Digest: {digest[:16]} | Status: {status}")
        if status != "100% PASS":
            raise RuntimeError("Integrity violation: baseline signature modified.")


# =====================================================================
# 5. DATASET & EVALUATION
# =====================================================================
class TaskDataset(Dataset):
    def __init__(self, dataset, tokenizer, allowed_labels, max_length=MAX_LENGTH):
        label_mapping = {orig: new for new, orig in enumerate(allowed_labels)}
        self.filtered_data = [
            (text, label_mapping[label])
            for text, label in zip(dataset['text'], dataset['label'])
            if label in label_mapping
        ]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.filtered_data)

    def __getitem__(self, idx):
        text, label = self.filtered_data[idx]
        encoding = self.tokenizer(
            text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long),
        }

    def majority_share(self):
        ones = sum(label for _, label in self.filtered_data)
        return max(ones, len(self) - ones) / len(self)


@torch.no_grad()
def evaluate_task(model, dataset, task, input_device, batch_size=32):
    """FIX: held-out accuracy and loss for one task head. The original never used the test split."""
    previous_task, was_training = model.current_task, model.training
    model.eval()
    model.switch_task(task)
    correct, total, loss_sum = 0, 0, 0.0
    for batch in DataLoader(dataset, batch_size=batch_size, shuffle=False):
        out = model(
            input_ids=batch['input_ids'].to(input_device),
            attention_mask=batch['attention_mask'].to(input_device),
            labels=batch['labels'],
        )
        preds = out['logits'].argmax(dim=-1).cpu()
        correct += (preds == batch['labels']).sum().item()
        total += batch['labels'].numel()
        loss_sum += out['loss'].item() * batch['labels'].numel()
    model.switch_task(previous_task)
    model.train(was_training)
    return correct / total, loss_sum / total


# =====================================================================
# 6. LOAD SUBSTRATE & DATA
# =====================================================================
print(f"Loading {MODEL_ID} ...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False
base_model.eval()
for param in base_model.parameters():
    param.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

embedding_layer = base_model.get_input_embeddings()
input_device = embedding_layer.weight.device
hidden_size = embedding_layer.weight.shape[1]
substrate_dtype = str(embedding_layer.weight.dtype).replace("torch.", "")
print(f"Substrate hidden size: {hidden_size} | weight dtype: {substrate_dtype}")

# Baseline digest taken from the live weights before any optimizer exists
BASELINE_DIGEST = anchor_digest(embedding_layer, PRIME_ANCHORS)
print(f"Baseline anchor digest: {BASELINE_DIGEST[:16]}")

model = GPTOSS_TaskAwareModel(base_model, hidden_size, DEVICE)

print("Loading 'SetFit/ag_news' ...")
raw_train = load_dataset("SetFit/ag_news", split=TRAIN_SPLIT)
raw_test = load_dataset("SetFit/ag_news", split=TEST_SPLIT)
train_datasets = {t: TaskDataset(raw_train, tokenizer, TASKS[t]['labels']) for t in TASK_ORDER}
test_datasets = {t: TaskDataset(raw_test, tokenizer, TASKS[t]['labels']) for t in TASK_ORDER}
majority = {t: test_datasets[t].majority_share() for t in TASK_ORDER}
for t in TASK_ORDER:
    print(f"  Task {t} ({TASKS[t]['names'][0]} vs {TASKS[t]['names'][1]}): "
          f"{len(train_datasets[t])} train / {len(test_datasets[t])} test | "
          f"majority-class accuracy {majority[t]:.2%}")

# =====================================================================
# 7. SEQUENTIAL TRAINING A -> B -> C WITH EVALUATION AFTER EVERY TASK
# =====================================================================
training_args = TrainingArguments(
    output_dir=RUN_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=8,
    learning_rate=HEAD_LR,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none",
    use_cpu=(DEVICE.type == "cpu"),
    remove_unused_columns=False,
    bf16=False,            # FIX: backbone is already bf16; the heads must stay float32
    save_strategy="no",
    seed=SEED,             # FIX: Trainer otherwise reseeds with its default of 42
    data_seed=SEED,
)

trainer = TopoTrainer(
    model=model,
    args=training_args,
    train_dataset=train_datasets[TASK_ORDER[0]],
    embedding_layer=embedding_layer,
    prime_anchors=PRIME_ANCHORS,
    callbacks=[TopoTelemetryCallback()],
)


def evaluate_all(stage):
    row = {}
    for t in TASK_ORDER:
        acc, loss = evaluate_task(model, test_datasets[t], t, input_device)
        row[t] = {"accuracy": acc, "loss": loss}
    print(f"[Eval after {stage}] " + " | ".join(
        f"Task {t}: acc {row[t]['accuracy']:.2%}, loss {row[t]['loss']:.4f}" for t in TASK_ORDER))
    return row


eval_history = {"init": evaluate_all("init")}   # untrained heads: loss 0.6931 expected
train_loss = {}

for task_name in TASK_ORDER:
    model.switch_task(task_name)
    trainer.train_dataset = train_datasets[task_name]
    trainer.lr_scheduler = None                 # fresh learning-rate schedule per task
    print(f"\nTraining on Task {task_name} via HF Trainer...")
    train_loss[task_name] = trainer.train().training_loss
    eval_history[task_name] = evaluate_all(f"Task {task_name}")

# Forgetting: best accuracy a task ever had after it was trained, minus its final accuracy
final_stage = TASK_ORDER[-1]
forgetting = {}
for j, t in enumerate(TASK_ORDER[:-1]):
    best = max(eval_history[s][t]["accuracy"] for s in TASK_ORDER[j:-1])
    forgetting[t] = best - eval_history[final_stage][t]["accuracy"]
mean_forgetting = sum(forgetting.values()) / len(forgetting)
final_accuracy = {t: eval_history[final_stage][t]["accuracy"] for t in TASK_ORDER}

print("\n--- Test accuracy (rows: evaluated after, columns: task head) ---")
print("after   " + "".join(f"{'Task ' + t:>10}" for t in TASK_ORDER))
for stage in ["init"] + TASK_ORDER:
    print(f"{stage:<8}" + "".join(f"{eval_history[stage][t]['accuracy']:>10.2%}" for t in TASK_ORDER))
print("majority" + "".join(f"{majority[t]:>10.2%}" for t in TASK_ORDER))
print(f"Forgetting per task: " + ", ".join(f"{t} {v:+.2%}" for t, v in forgetting.items())
      + f" | mean {mean_forgetting:+.2%}")

# =====================================================================
# 8. INTEGRITY GATE & ARTIFACT EXPORT
# =====================================================================
topo_opt = unwrap_optimizer(trainer.optimizer)
assert isinstance(topo_opt, TopoAdamW), "Trainer did not use TopoAdamW."

print("\n--- Integrity gate ---")
max_drift, audited_digest, hash_matched = topo_opt.audit_telemetry()
live_digest = anchor_digest(embedding_layer, PRIME_ANCHORS)   # recomputed from the live weights
print(f"Baseline digest (pre-training) : {BASELINE_DIGEST[:16]}")
print(f"Optimizer baseline digest      : {topo_opt.certified_sha256_digest[:16]}")
print(f"Live digest (post-training)    : {live_digest[:16]}")
print(f"Max drift                      : {max_drift:.10f}")
assert hash_matched and max_drift == 0.0, "Gate failed: anchor rows changed."
assert live_digest == BASELINE_DIGEST == topo_opt.certified_sha256_digest, "Gate failed: digest mismatch."
print("Verdict                        : PASS - anchor rows are bit-identical to the baseline.")

os.makedirs(RUN_DIR, exist_ok=True)
checkpoint = {
    "task_heads_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()
                              if k.startswith("classifier_")},
    "state_hash": live_digest,
    "max_drift": max_drift,
    "base_model_id": MODEL_ID,
    "substrate_dtype": substrate_dtype,
    "hidden_size": hidden_size,
    "head_arch": "LayerNorm(hidden) -> Linear(hidden, 2), float32",
    "prime_anchors": PRIME_ANCHORS,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "task_order": TASK_ORDER,
    "tasks": TASKS,
    "train_split": TRAIN_SPLIT,
    "test_split": TEST_SPLIT,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": HEAD_LR,
    "train_loss": train_loss,
    "eval_history": eval_history,
    "majority_accuracy": majority,
    "final_accuracy": final_accuracy,
    "forgetting": forgetting,
    "mean_forgetting": mean_forgetting,
}
torch.save(checkpoint, CHECKPOINT_PATH)
print(f"Saved task heads, audit trail and measured results: {CHECKPOINT_PATH} "
      f"({os.path.getsize(CHECKPOINT_PATH) / 1024:.1f} kB)")

Loading openai/gpt-oss-20b ...


config.json:   0%|          | 0.00/1.81k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/36.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

test.jsonl: reconstructing file:   0%|          |  0.00B / 2.13MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

  Task A (World vs Sports): 1313 train / 1037 test | majority-class accuracy 50.72%
  Task B (Business vs Sci/Tech): 1687 train / 963 test | majority-class accuracy 53.37%
  Task C (World vs Sci/Tech): 1716 train / 1025 test | majority-class accuracy 50.15%
[Eval after init] Task A: acc 49.28%, loss 0.6931 | Task B: acc 46.63%, loss 0.6931 | Task C: acc 49.85%, loss 0.6931

Training on Task A via HF Trainer...
[TopoAdamW] Baseline Bedrock Locked. Certified Digest: 334ea0c8ca2e9af5
{'loss': '0.5779', 'grad_norm': '10.41', 'learning_rate': '0.0009891', 'epoch': '0.06061'}
{'loss': '0.2044', 'grad_norm': '3.772', 'learning_rate': '0.000977', 'epoch': '0.1212'}
{'loss': '0.4368', 'grad_norm': '8.788', 'learning_rate': '0.0009648', 'epoch': '0.1818'}
{'loss': '0.1394', 'grad_norm': '1.237', 'learning_rate': '0.0009527', 'epoch': '0.2424'}
{'loss': '0.2685', 'grad_norm': '0.8594', 'learning_rate': '0.0009406', 'epoch': '0.303'}
{'loss': '0.2437', 'grad_norm': '1.715', 'learning_rate': '0.000

### 2. Build the model card from the measured results and publish

In [ ]:
import os
import torch
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

# =====================================================================
# CONFIGURATION
# =====================================================================
HF_TOKEN = userdata.get('HF_TOKEN')
USER_OR_ORG = 'frankmorales2020'
REPO_NAME = "gpt-oss-20b-topo-governed-cbp-new-version2"
HF_REPO_ID = f"{USER_OR_ORG}/{REPO_NAME}"
CHECKPOINT_PATH = "/content/drive/MyDrive/topogpt/gpt_oss_20b_hf_topo_run/certified_topological_best.pt"

assert os.path.exists(CHECKPOINT_PATH), f"{CHECKPOINT_PATH} not found - run the training cell first."
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")

# =====================================================================
# MODEL CARD (README.md) - FIX: every number below is read from the checkpoint
# =====================================================================
order = ckpt["task_order"]
tasks = ckpt["tasks"]
hist = ckpt["eval_history"]
stages = ["init"] + order

task_rows = "\n".join(
    f"| {t} | {tasks[t]['names'][0]} vs {tasks[t]['names'][1]} | {ckpt['train_loss'][t]:.4f} | "
    f"{hist[order[-1]][t]['loss']:.4f} | {ckpt['final_accuracy'][t]:.2%} | {ckpt['majority_accuracy'][t]:.2%} |"
    for t in order
)
matrix_rows = "\n".join(
    f"| {s} | " + " | ".join(f"{hist[s][t]['accuracy']:.2%}" for t in order) + " |"
    for s in stages
)
forgetting_line = ", ".join(f"Task {t}: {v:+.2%}" for t, v in ckpt["forgetting"].items())
tags = ["topological-continual-backpropagation", "topological-anchoring", "gpt-oss-20b", "huggingface-trainer"]
if ckpt["mean_forgetting"] <= 0:
    tags.append("zero-forgetting")
tag_block = "\n".join(f"- {tag}" for tag in tags)

readme_content = f"""---
license: apache-2.0
tags:
{tag_block}
base_model: {ckpt['base_model_id']}
---

# GPT-OSS-20B: Topo-CBP task heads trained with the Hugging Face Trainer

Three binary AG News classification heads trained in sequence (A -> B -> C) on top of a frozen
`{ckpt['base_model_id']}` backbone, using `TopoAdamW` inside a subclassed Hugging Face `Trainer`.

## Setup
- **Backbone:** `{ckpt['base_model_id']}`, frozen, weights loaded as `{ckpt['substrate_dtype']}`, hidden size {ckpt['hidden_size']}.
- **Heads:** one per task, `{ckpt['head_arch']}`, fed the hidden state of the last real token.
- **Optimizer:** `TopoAdamW`, learning rate {ckpt['learning_rate']}, {ckpt['num_epochs']} epochs per task, seed {ckpt['seed']}.
- **Data:** `SetFit/ag_news`, `{ckpt['train_split']}` for training and `{ckpt['test_split']}` for evaluation, max length {ckpt['max_length']}.

## Results on the held-out test split
| Task | Classes | Mean train loss | Final test loss | Final test accuracy | Majority-class accuracy |
|---|---|---|---|---|---|
{task_rows}

Test accuracy of every head after each training stage:

| Evaluated after | {" | ".join("Task " + t for t in order)} |
|---|{"---|" * len(order)}
{matrix_rows}

- **Forgetting** (best accuracy after a task was trained, minus its final accuracy): {forgetting_line}
- **Mean forgetting:** {ckpt['mean_forgetting']:+.2%}

## Integrity audit
- **Anchor rows:** token-embedding rows {ckpt['prime_anchors']}
- **SHA-256 digest of the anchor rows:** `{ckpt['state_hash'][:16]}` (identical before and after training)
- **Max drift on the anchor rows:** `{ckpt['max_drift']:.10f}`

## Scope
The backbone is frozen and each task has its own head, so zero drift and zero forgetting follow
from the design. The figures above confirm the pipeline behaves as designed; they do not compare
`TopoAdamW` against plain AdamW.

## Checkpoint
`certified_topological_best.pt` holds the three task heads, the task and label definitions,
the anchor digest and the measured results shown above.
"""

with open("README.md", "w") as f:
    f.write(readme_content)

api = HfApi(token=HF_TOKEN)
print(f"Creating or verifying repository: {HF_REPO_ID}...")
create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, token=HF_TOKEN)

for local_path, repo_path in [("README.md", "README.md"), (CHECKPOINT_PATH, "certified_topological_best.pt")]:
    print(f"Uploading {repo_path} to {HF_REPO_ID}...")
    api.upload_file(path_or_fileobj=local_path, path_in_repo=repo_path, repo_id=HF_REPO_ID, repo_type="model")

print(f"Published to: https://huggingface.co/{HF_REPO_ID}")

Creating or verifying repository: frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2...
Uploading README.md to frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2...
Uploading certified_topological_best.pt to frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ified_topological_best.pt: 100%|##########|  145kB /  145kB            

Validating                    :           |   0%            

Published to: https://huggingface.co/frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2


### 3. Independent reload: verify the digest, re-measure test accuracy, run example queries

In [ ]:
import gc
import hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer

# =====================================================================
# 1. SEED & CONFIGURATION (independent of the training cell)
# =====================================================================
SEED = 123
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GOVERNED_REPO_ID = "frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Free the training session's copy of the backbone so it is not held in VRAM twice
for _name in ("trainer", "topo_opt", "model", "base_model", "embedding_layer"):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =====================================================================
# 2. DOWNLOAD THE CHECKPOINT AND LOAD THE BASE MODEL IT NAMES
# =====================================================================
print(f"Downloading checkpoint from {GOVERNED_REPO_ID}...")
checkpoint_file = hf_hub_download(repo_id=GOVERNED_REPO_ID, filename="certified_topological_best.pt")
ckpt = torch.load(checkpoint_file, map_location="cpu")

BASE_MODEL_ID = ckpt["base_model_id"]
TASK_ORDER = ckpt["task_order"]
TASKS = ckpt["tasks"]          # FIX: label names come from the checkpoint, not a second hand-typed table
MAX_LENGTH = ckpt["max_length"]

print(f"Loading base substrate: {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, device_map="auto", dtype=torch.bfloat16, low_cpu_mem_usage=True,
)
base_model.eval()
embedding_layer = base_model.get_input_embeddings()
input_device = embedding_layer.weight.device
hidden_size = embedding_layer.weight.shape[1]

# =====================================================================
# 3. INTEGRITY AUDIT - FIX: digest recomputed from the weights just loaded
# =====================================================================
def anchor_digest(embedding_layer, anchors):
    w = embedding_layer.weight.data
    rows = [w[i] for i in anchors if i < w.shape[0]]
    blob = torch.cat([r.flatten().float() for r in rows]).detach().cpu().numpy().tobytes()
    return hashlib.sha256(blob).hexdigest()

recomputed = anchor_digest(embedding_layer, ckpt["prime_anchors"])
print(f"Digest stored in checkpoint  : {ckpt['state_hash'][:16]}")
print(f"Digest recomputed from model : {recomputed[:16]}")
assert recomputed == ckpt["state_hash"], "Integrity failure: loaded base model does not match the checkpoint digest."
assert hidden_size == ckpt["hidden_size"], "Hidden size mismatch between base model and checkpoint."
print("Integrity audit              : PASS - anchor rows match the trained-against baseline.\n")

# =====================================================================
# 4. REBUILD THE HEADS AND BIND THE WEIGHTS (strict)
# =====================================================================
def make_head(hidden_size):
    return nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, 2)).float()

heads = nn.ModuleDict({f"classifier_{t}": make_head(hidden_size) for t in TASK_ORDER})
heads.load_state_dict(ckpt["task_heads_state_dict"], strict=True)   # FIX: was strict=False
heads.to(DEVICE).eval()


def pool_last_token(hidden_states, attention_mask):
    positions = torch.arange(attention_mask.shape[1], device=attention_mask.device)
    last_idx = (attention_mask.long() * positions).argmax(dim=1)
    batch_idx = torch.arange(hidden_states.shape[0], device=hidden_states.device)
    return hidden_states[batch_idx, last_idx.to(hidden_states.device), :]


@torch.no_grad()
def predict(texts, task):
    enc = tokenizer(texts, max_length=MAX_LENGTH, padding="max_length", truncation=True, return_tensors="pt")
    enc = {k: v.to(input_device) for k, v in enc.items()}
    out = base_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                     output_hidden_states=True, return_dict=True)
    pooled = pool_last_token(out.hidden_states[-1], enc["attention_mask"])
    with torch.autocast(device_type=DEVICE.type, enabled=False):
        logits = heads[f"classifier_{task}"](pooled.to(device=DEVICE, dtype=torch.float32))
    return F.softmax(logits, dim=-1).cpu()

# =====================================================================
# 5. HELD-OUT ACCURACY - FIX: measured against ground truth, compared with the stored figure
# =====================================================================
raw_test = load_dataset("SetFit/ag_news", split=ckpt["test_split"])
print(f"--- Test accuracy on SetFit/ag_news {ckpt['test_split']} ---")
for task in TASK_ORDER:
    mapping = {orig: new for new, orig in enumerate(TASKS[task]["labels"])}
    pairs = [(t, mapping[l]) for t, l in zip(raw_test["text"], raw_test["label"]) if l in mapping]
    correct = 0
    for i in range(0, len(pairs), 32):
        chunk = pairs[i:i + 32]
        probs = predict([t for t, _ in chunk], task)
        correct += (probs.argmax(dim=-1) == torch.tensor([y for _, y in chunk])).sum().item()
    acc = correct / len(pairs)
    print(f"Task {task} ({TASKS[task]['names'][0]} vs {TASKS[task]['names'][1]}): "
          f"{acc:.2%} on {len(pairs)} samples | recorded at training time: {ckpt['final_accuracy'][task]:.2%} | "
          f"majority class: {ckpt['majority_accuracy'][task]:.2%}")

# =====================================================================
# 6. EXAMPLE QUERIES - FIX: each query belongs to its task and has an expected label
# =====================================================================
queries = {
    'A': [("International diplomatic summits opened in Geneva to discuss regional peace treaties.", "World"),
          ("The national football championship ended in an overtime thriller last night.", "Sports")],
    'B': [("Wall Street indexes rallied to record highs following positive corporate earnings.", "Business"),
          ("A revolutionary quantum computing chip demonstrates exponential speedup in cryptographic calculations.", "Sci/Tech")],
    'C': [("Peace negotiators from both governments met in the capital to discuss a ceasefire.", "World"),
          ("Space exploration agencies successfully placed a new observation satellite into orbit.", "Sci/Tech")],
}

print()
for task in TASK_ORDER:
    names = TASKS[task]["names"]
    print(f"--- Task head {task}: {names[0]} (class 0) vs {names[1]} (class 1) ---")
    for text, expected in queries[task]:
        probs = predict([text], task)[0]
        pred = int(probs.argmax())
        verdict = "correct" if names[pred] == expected else "WRONG"
        print(f"Query      : \"{text}\"")
        print(f"Prediction : {names[pred]} (class {pred}) | confidence {probs[pred].item():.2%} | "
              f"expected {expected} -> {verdict}\n")

certified_topological_best.pt: reconstructing file:   0%|          |  0.00B /  145kB            

certified_topological_best.pt: downloading bytes:           |  0.00B            

Loading base substrate: openai/gpt-oss-20b...


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

Digest stored in checkpoint  : 334ea0c8ca2e9af5
Digest recomputed from model : 334ea0c8ca2e9af5
Integrity audit              : PASS - anchor rows match the trained-against baseline.

--- Test accuracy on SetFit/ag_news test[:2000] ---
Task A (World vs Sports): 96.34% on 1037 samples | recorded at training time: 96.34% | majority class: 50.72%
Task B (Business vs Sci/Tech): 88.58% on 963 samples | recorded at training time: 88.58% | majority class: 53.37%
Task C (World vs Sci/Tech): 95.41% on 1025 samples | recorded at training time: 95.41% | majority class: 50.15%

--- Task head A: World (class 0) vs Sports (class 1) ---
Query      : "International diplomatic summits opened in Geneva to discuss regional peace treaties."
Prediction : World (class 0) | confidence 100.00% | expected World -> correct

Query      : "The national football championship ended in an overtime thriller last night."
Prediction : Sports (class 1) | confidence 99.49% | expected Sports -> correct

--- Task head B: Bu